In [6]:
import os
import pandas as pd

os.makedirs("data/raw", exist_ok=True)

# 1. Load the 3 raw CPDB files
# keep_default_na=False keeps chemcode 'na' (5-NITRO-o-ANISIDINE) from turning into NaN
lit = pd.read_excel("../data/raw/cpdb.lit.xls", keep_default_na=False)
chem = pd.read_excel("../data/raw/cpdb.chemname.xls", keep_default_na=False)
dose = pd.read_excel("../data/raw/cpdb.litdose.xls", keep_default_na=False)

lit.columns = lit.columns.str.strip().str.lower()
chem.columns = chem.columns.str.strip().str.lower()
dose.columns = dose.columns.str.strip().str.lower()

# 2. Aggregate cpdb.litdose.xls to 1 row per test (idnum)
dose["dose"] = pd.to_numeric(dose["dose"], errors="coerce")
dose_agg = (
    dose[dose["dose"] > 0]
    .groupby("idnum", as_index=False)
    .agg(
        max_dose=("dose", "max"),
        min_dose=("dose", "min"),
        mean_dose=("dose", "mean"),
        num_dose_groups=("dose", "count"),
    )
)

# 3. Merge into one unified dataset and save untouched
chem_unique = chem[["chemcode", "name", "cas"]].drop_duplicates(subset=["chemcode"])
merged = pd.merge(lit, chem_unique, on="chemcode", how="left")
merged = pd.merge(merged, dose_agg, on="idnum", how="left")

merged.to_csv("../data/processed/cp_data.csv", index=False)
print(f"Step 1 complete! Saved {len(merged)} merged rows to data/processed/cp_dataset.csv")

/var/folders/sl/hdz3978x7kzf7f1wy3vcc7140000gn/T/ipykernel_3264/1173929136.py:8: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  lit = pd.read_excel("../data/raw/cpdb.lit.xls", keep_default_na=False)
/var/folders/sl/hdz3978x7kzf7f1wy3vcc7140000gn/T/ipykernel_3264/1173929136.py:9: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  chem = pd.read_excel("../data/raw/cpdb.chemname.xls", keep_default_na=False)
/var/folders/sl/hdz3978x7kzf7f1wy3vcc7140000gn/T/ipykernel_3264/1173929136.py:10: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  dose = pd.read_excel("../data/raw/cpdb.litdose.xls", keep_default_na=False)


Step 1 complete! Saved 14207 merged rows to data/processed/cp_dataset.csv
